# LAB | Micromanage your agents

**Scenario:** you are building an evaluation system for **one specific use case**. Public benchmarks (like HLE in class) don't reflect your client's real questions, so you build **your own dataset**, upload it to LangSmith, run experiments and report what you found.

**How to use this notebook:**
- Cells marked **`# TODO`** are yours to complete. Replace every `...` with your code.
- **Hints** point to the lesson notebook (`02_langsmith(1).ipynb`), where you saw each idea.
- Markdown cells with a bold prompt (e.g. **Document your choice here:**, **Key findings:**) are for your written answers.
- Run the cells **in order**, top to bottom.

| Part | You will… | Concept from class |
|---|---|---|
| 1 | Choose a domain + set up LangSmith | Tracing, projects, regions |
| 2 | Build and upload your own dataset | Datasets: inputs, reference outputs, metadata |
| 3 | Write the target function + evaluators | Target, LLM-as-a-judge (OpenEvals), exact match |
| 4 | Run the experiment and review it in the UI | `client.evaluate()`, experiments, feedback |
| 5 | Analyse results + write `evaluation_summary.md` | Metrics, error analysis |
| Optional 1 | Design a custom evaluator | Judge prompt anatomy (notebook 01) |
| Optional 2 | A/B test models on cost vs performance | Model comparison, cost per correct answer |

In [ ]:
%pip install langsmith openai openevals python-dotenv pandas matplotlib -q

## Part 1: Setup & document selection (30–45 min)
### Step 1: Choose your domain and documents

Pick a domain where **you** can judge what a good answer looks like. Ideas: customer support FAQ, a company policy, a product catalogue, course material, a legal or compliance text, code review…

Each example needs:
- a clear **input** (question, ticket, task)
- ideally an **expected output** (reference answer, label, summary)
- enough **variety**: easy and hard cases, different categories

 **Hint:** the best datasets include **tricky cases**: questions with no answer in the document, ambiguous wording, multi-step questions. Easy-only datasets give 100% scores and teach you nothing (remember the calibration set in notebook 01, §6).

** Document your choice here:**
- Domain (2–3 sentences): Northstar Outfitters customer-support policy Q&A. The system must answer shoppers using only a concise fictional policy covering returns, shipping, warranty, and support hours.
- Source(s) of my documents (name + URL or file): `policy.md`, a versioned policy document created for this lab.
- What is one example? (input → expected output): `Is express shipping free on a $100 order?` → `No. Orders over $75 get free standard shipping, but the $18 express surcharge still applies.`
- What makes a *good* answer in my domain? It must be factually grounded in the policy, answer every part of the question, preserve important numbers and conditions, and explicitly say when the policy does not state something.

### Step 2: Set up LangSmith

 **Hint:** copy the setup pattern from `02_langsmith(1).ipynb` (Setup: keys and region). Your `.env` needs `LANGSMITH_API_KEY` and `OPENAI_API_KEY`. Check your **region**: `eu.smith.langchain.com` in the browser means EU.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Turn tracing on, use a descriptive project name, and target the EU endpoint.
os.environ["LANGSMITH_TRACING"] = "true"
PROJECT_NAME = "lab-langsmith-support-policy-v1"
os.environ["LANGSMITH_PROJECT"] = PROJECT_NAME
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from langsmith import Client, traceable
from langsmith.wrappers import wrap_openai
from openai import OpenAI
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT

OUT = Path("outputs"); OUT.mkdir(exist_ok=True)   # charts and CSVs are saved here

# Create the LangSmith client and a WRAPPED OpenAI client (so every call is traced).
client = Client()
openai_client = wrap_openai(OpenAI())

# Verify the connection without assuming that any dataset already exists.
datasets = list(client.list_datasets())
print(f"LangSmith connection OK; visible datasets: {len(datasets)}")

## Part 2: Dataset creation (60–90 min)
### Step 3: Extract and structure your examples

Turn your documents into a list of examples. Every example is a dictionary with the **same keys**.

| Field | Goes to | Example |
|---|---|---|
| `question` (+ `context` if RAG-style) | **inputs**: what the model sees | "Can I return a sale item?" |
| `answer` | **outputs**: the reference answer | "Yes, within 14 days, store credit only." |
| `category`, `difficulty`, `source` | **metadata**: for analysis, the model never sees it | "returns", "hard", "policy.pdf p.3" |

 **Hint:** aim for **15–20 examples** (minimum 10). Include at least 2 categories and at least 2 "hard" or "trap" examples.

In [ ]:
# The source is a versioned local policy document, loaded once and supplied as
# context to the target. References are intentionally concise and auditable.
POLICY_CONTEXT = Path("policy.md").read_text(encoding="utf-8")
SOURCE = "policy.md (Northstar Outfitters Customer Support Policy v1)"
examples = [
    {"question": "Can I return an unopened item 20 days after delivery?", "context": POLICY_CONTEXT, "answer": "Yes. An unopened, unused item may be returned within 30 calendar days of delivery for a full refund to the original payment method.", "category": "returns", "difficulty": "easy", "source": SOURCE},
    {"question": "I opened an item and changed my mind. Can I return it after 10 days?", "context": POLICY_CONTEXT, "answer": "No. Opened items may be returned within 14 days only when they are defective or damaged on arrival, with a photo of the defect.", "category": "returns", "difficulty": "medium", "source": SOURCE},
    {"question": "Are sale items eligible for a cash refund?", "context": POLICY_CONTEXT, "answer": "No. Sale items are exchange-only within 14 calendar days and are not eligible for a cash refund.", "category": "returns", "difficulty": "easy", "source": SOURCE},
    {"question": "Can I exchange a personalized item?", "context": POLICY_CONTEXT, "answer": "No. Personalized items are final sale and cannot be returned or exchanged.", "category": "returns", "difficulty": "easy", "source": SOURCE},
    {"question": "Who pays return shipping when the item is defective?", "context": POLICY_CONTEXT, "answer": "Northstar pays return shipping for defective or damaged items.", "category": "returns", "difficulty": "easy", "source": SOURCE},
    {"question": "How long can a refund take after inspection?", "context": POLICY_CONTEXT, "answer": "Northstar issues the refund within 5 business days after inspection; the bank may take an additional 3–5 business days to display it.", "category": "returns", "difficulty": "medium", "source": SOURCE},
    {"question": "What does standard shipping cost and how long does it take?", "context": POLICY_CONTEXT, "answer": "Standard shipping costs $6 and usually arrives in 3–5 business days.", "category": "shipping", "difficulty": "easy", "source": SOURCE},
    {"question": "What does express shipping cost?", "context": POLICY_CONTEXT, "answer": "Express shipping costs $18 and usually arrives in 1–2 business days.", "category": "shipping", "difficulty": "easy", "source": SOURCE},
    {"question": "My order is $100. Is express shipping free?", "context": POLICY_CONTEXT, "answer": "No. Orders over $75 qualify for free standard shipping, but the $18 express surcharge still applies.", "category": "shipping", "difficulty": "hard", "source": SOURCE},
    {"question": "Can I cancel an order after it has shipped?", "context": POLICY_CONTEXT, "answer": "Not through cancellation. An order can be canceled for a full refund before it ships; after shipment, use the returns process.", "category": "shipping", "difficulty": "hard", "source": SOURCE},
    {"question": "How long is the warranty for manufacturing defects?", "context": POLICY_CONTEXT, "answer": "Manufacturing defects are covered for 12 months after delivery.", "category": "warranty", "difficulty": "easy", "source": SOURCE},
    {"question": "Is accidental damage covered by the warranty?", "context": POLICY_CONTEXT, "answer": "No. The warranty excludes accidental damage, normal wear, and misuse.", "category": "warranty", "difficulty": "medium", "source": SOURCE},
    {"question": "What information is needed for a warranty claim?", "context": POLICY_CONTEXT, "answer": "A warranty claim requires the order number and a description of the problem; photos or a short video may also be requested.", "category": "warranty", "difficulty": "medium", "source": SOURCE},
    {"question": "When is customer support available?", "context": POLICY_CONTEXT, "answer": "Support is available Monday–Friday, 09:00–17:00 Central European Time.", "category": "support", "difficulty": "easy", "source": SOURCE},
    {"question": "Does the policy promise Saturday delivery?", "context": POLICY_CONTEXT, "answer": "No. The policy does not promise Saturday delivery.", "category": "policy-boundary", "difficulty": "hard", "source": SOURCE},
    {"question": "Do you offer price matching?", "context": POLICY_CONTEXT, "answer": "The policy does not state that Northstar offers price matching.", "category": "policy-boundary", "difficulty": "hard", "source": SOURCE},
]

In [ ]:
assert len(examples) >= 10, "The dataset must contain at least 10 examples"
keys = set(examples[0])
assert all(set(ex) == keys for ex in examples), "Every example must have the same keys"
assert all(ex["question"].strip() and ex["answer"].strip() for ex in examples)
assert {ex["category"] for ex in examples} >= {"returns", "shipping"}
assert sum(ex["difficulty"] == "hard" for ex in examples) >= 2
print(f"Validated {len(examples)} examples; categories:")
print(pd.Series([ex["category"] for ex in examples]).value_counts().sort_index())
print("Difficulty:")
print(pd.Series([ex["difficulty"] for ex in examples]).value_counts().sort_index())

** My data structure:** `question` and `context` are the model inputs; `answer` is the reference output; `category`, `difficulty`, and `source` are metadata used only for analysis. I used the complete `policy.md` as context for every example and kept 16 cases across six categories, including four hard conditional or policy-boundary cases.

### Step 4: Create the LangSmith dataset

 **Hints (§2 of `02_langsmith(1).ipynb`):**
- Check `client.list_datasets()` first, so re-running the cell doesn't create a duplicate.
- `client.create_dataset(name, description=...)` then `client.create_examples(inputs=[...], outputs=[...], metadata=[...], dataset_id=...)`.
- Put a **version** in the name (e.g. `-v1`). If you change the examples later, create `-v2` instead of editing silently.

In [ ]:
DATASET_NAME = "northstar-support-policy-v1"
DATASET_DESCRIPTION = (
    "16 grounded customer-support questions and reference answers derived from policy.md; "
    "metadata records category, difficulty, and source."
)

# Create the dataset only if it doesn't exist yet, then upload inputs, outputs, and metadata.
existing = {d.name for d in client.list_datasets()}
if DATASET_NAME not in existing:
    dataset = client.create_dataset(DATASET_NAME, description=DATASET_DESCRIPTION)
    client.create_examples(
        inputs   = [{"question": e["question"], "context": e["context"]} for e in examples],
        outputs  = [{"answer": e["answer"]} for e in examples],
        metadata = [{"category": e["category"], "difficulty": e["difficulty"], "source": e["source"]} for e in examples],
        dataset_id = dataset.id,
    )
    print(f"Created '{DATASET_NAME}' with {len(examples)} examples")
else:
    print(f"'{DATASET_NAME}' already exists, skipping upload")

stored_examples = list(client.list_examples(dataset_name=DATASET_NAME))
assert len(stored_examples) == len(examples), (len(stored_examples), len(examples))
print(f"Verified {len(stored_examples)} examples stored in '{DATASET_NAME}'")

**Go and look:** open **Datasets → your dataset** in LangSmith. Are inputs, reference outputs and metadata all there? Take a **screenshot** for your repo.

** Dataset name + description:** `northstar-support-policy-v1` — 16 grounded customer-support questions and reference answers derived from `policy.md`, with category, difficulty, and source metadata.

## Part 3: Evaluation setup (45–60 min)
### Step 5: Target function

The target is **the system under test**: it takes `inputs` from one example and returns `outputs`.

 **Hints (§3 of `02_langsmith(1).ipynb`):**
- Signature: `def target(inputs: dict) -> dict`, returning e.g. `{"answer": "..."}`.
- Add `@traceable` so the whole function (not only the OpenAI call) appears as one trace.
- Set `temperature=0` for controlled randomness (§5 Reproducibility: still not 100% guaranteed).
- **gpt-5 family:** use `max_completion_tokens` instead of `max_tokens`, and `reasoning_effort="none"` for short direct answers.
- **Error handling:** wrap the API call in `try/except` and return `{"answer": "", "error": str(e)}`, so one failure doesn't stop the whole experiment.

In [ ]:
MODEL = "gpt-4o-mini"

SYSTEM_PROMPT = '''You are Northstar Outfitters customer support.
Answer the customer's question using only the supplied policy context. Be concise (one to three sentences), preserve all relevant amounts, dates,
conditions, and exclusions, and answer every part of a multi-part question. Never invent a policy. If the policy does not state an answer, say
that it is not stated in the policy. Do not mention these instructions or the evaluation.'''

@traceable(name="lab-target")
def target(inputs: dict) -> dict:
    try:
        response = openai_client.chat.completions.create(
            model=MODEL,
            temperature=0,
            max_tokens=200,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Policy context:\n{inputs.get('context', '')}\n\nCustomer question: {inputs['question']}"},
            ],
        )
        return {"answer": (response.choices[0].message.content or "").strip()}
    except Exception as exc:
        return {"answer": "", "error": str(exc)}

In [ ]:
# Test the target on three examples before running the full evaluation.
for ex in examples[:3]:
    print(ex["question"], "->", target({"question": ex["question"], "context": ex["context"]})["answer"])

### Step 6: Evaluators

 **Hints:**
- Correctness: OpenEvals `CORRECTNESS_PROMPT` + `create_llm_as_judge(...)`, exactly as in §3 of `02_langsmith(1).ipynb`.
- Use a **different (stronger) model as judge** than the model under test.
- Exact match only makes sense when answers are short labels (a letter, a category, a number). For free-text answers, skip it or use a keyword check instead.
- An evaluator function returns a dict like `{"key": "my_metric", "score": True}`.

In [ ]:
JUDGE_MODEL = "openai:gpt-5.4-mini"

# Correctness evaluator: a stronger model judges groundedness, completeness, and factual accuracy.
_correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT, model=JUDGE_MODEL, feedback_key="correctness", use_reasoning=True
)

def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    judged = _correctness_judge(
        inputs=inputs, outputs=outputs, reference_outputs=reference_outputs
    )
    return {"key": "correctness", "score": bool(judged.get("score", False)), "comment": judged.get("comment", "")}

# Cheap deterministic evaluator: meaningful-word coverage plus preservation of numeric constraints.
import re
STOPWORDS = {"a", "an", "and", "are", "be", "can", "for", "from", "in", "is", "it", "may", "no", "of", "on", "or", "the", "to", "within", "with"}
def rule_evaluator(outputs: dict, reference_outputs: dict) -> dict:
    answer = outputs.get("answer", "").lower()
    reference = reference_outputs.get("answer", "").lower()
    ref_words = set(re.findall(r"[a-z0-9]+", reference)) - STOPWORDS
    answer_words = set(re.findall(r"[a-z0-9]+", answer))
    coverage = len(ref_words & answer_words) / max(len(ref_words), 1)
    required_numbers = set(re.findall(r"\d+(?:[–-]\d+)?", reference))
    numbers_present = all(n in answer for n in required_numbers)
    passed = bool(answer.strip()) and coverage >= 0.45 and numbers_present
    return {"key": "keyword_coverage", "score": passed, "comment": f"coverage={coverage:.2f}; required_numbers_present={numbers_present}"}

In [ ]:
reference = {"answer": examples[0]["answer"]}
good = {"answer": examples[0]["answer"]}
bad = {"answer": "Yes, express shipping is free and arrives tomorrow."}
print("Good rule result:", rule_evaluator(good, reference))
print("Bad rule result:", rule_evaluator(bad, reference))
print("Good judge result:", correctness_evaluator({"question": examples[0]["question"]}, good, reference))
print("Bad judge result:", correctness_evaluator({"question": examples[0]["question"]}, bad, reference))

** What does "correct" mean in my domain?** One correct answer or many valid ones? What does each evaluator measure?

## Part 4: Run the evaluation (30–45 min)
### Step 7: Execute the experiment

 **Hint (§3 of `02_langsmith(1).ipynb`):** `client.evaluate(target, data=..., evaluators=[...], experiment_prefix=..., max_concurrency=2)`. Use a prefix that tells you **what** ran, e.g. `"gpt-4o-mini-promptv1"`.

In [ ]:
results = client.evaluate(
    target,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator, rule_evaluator],
    experiment_prefix="gpt-4o-mini-support-policy-v1",
    max_concurrency=2,
)
print(results)

In [ ]:
# Collect the results and normalize feedback columns for the analysis cells below.
df = results.to_pandas()
print(df.columns.tolist())
assert len(df) == len(examples), f"Expected {len(examples)} rows, got {len(df)}"
def feedback_column(key):
    candidates = [c for c in df.columns if "feedback" in c.lower() and key in c.lower()]
    if not candidates:
        raise KeyError(f"No feedback column found for {key}: {df.columns.tolist()}")
    return candidates[0]
def value_column(fragment):
    candidates = [c for c in df.columns if fragment in c.lower()]
    return candidates[0] if candidates else None
correctness_col = feedback_column("correctness")
rule_col = feedback_column("keyword_coverage")
df["correctness_score"] = pd.to_numeric(df[correctness_col], errors="coerce").fillna(False).astype(float)
df["rule_score"] = pd.to_numeric(df[rule_col], errors="coerce").fillna(False).astype(float)
print(f"Processed all {len(df)} dataset examples")
df.to_csv(OUT / "results_main.csv", index=False)

### Step 8: Review the results in the LangSmith UI

**Go and look:** click the experiment link printed above.
- Open 2–3 **failed** examples and read the judge's reasoning (`comment`).
- Do the LLM judge and your rule-based evaluator **agree**? Look at the rows where they don't.
- Take **screenshots** (experiment table + one trace) for your repo.

** First observations + experiment link:**

## Part 5: Analysis & reporting (45–60 min)
### Step 9: Analyse results

In [ ]:
metric_rows = []
for name, column in {"correctness_llm": "correctness_score", "keyword_coverage": "rule_score"}.items():
    scores = df[column].dropna().astype(float)
    metric_rows.append({"metric": name, "n": len(scores), "pass_rate": scores.mean(), "failed": int((scores < 1).sum())})
metrics = pd.DataFrame(metric_rows)
print(metrics.to_string(index=False))
metrics.to_csv(OUT / "metrics_main.csv", index=False)

In [ ]:
# Performance by category and by difficulty.
# The results DataFrame has no metadata, so join it from the dataset:
meta = pd.DataFrame([
    {"example_id": str(e.id), **(e.metadata or {})}
    for e in client.list_examples(dataset_name=DATASET_NAME)
])
df["example_id"] = df["example_id"].astype(str)
df = df.merge(meta, on="example_id", how="left")

category_scores = df.groupby("category", dropna=False)[["correctness_score", "rule_score"]].mean().round(3)
difficulty_scores = df.groupby("difficulty", dropna=False)[["correctness_score", "rule_score"]].mean().round(3)
print("By category:\n", category_scores)
print("By difficulty:\n", difficulty_scores)
category_scores.to_csv(OUT / "pass_rate_by_category.csv")
difficulty_scores.to_csv(OUT / "pass_rate_by_difficulty.csv")
ax = category_scores.plot(kind="bar", ylim=(0, 1.05), figsize=(8, 4), title="Pass rate by category")
ax.set_ylabel("Pass rate"); ax.set_xlabel("Category"); plt.xticks(rotation=30, ha="right"); plt.tight_layout()
plt.savefig(OUT / "pass_rate_by_category.png", dpi=120); plt.show()

In [ ]:
# Error analysis: preserve the raw question/answer/reference/comment fields and add a
# small, transparent hand-coded taxonomy for the cases that failed the LLM judge.
def find_col(fragment, default=""):
    matches = [c for c in df.columns if fragment in c.lower()]
    return matches[0] if matches else default
question_col = find_col("inputs.question") or find_col("question")
answer_col = find_col("outputs.answer") or find_col("answer")
reference_col = find_col("reference.answer") or find_col("reference")
comment_candidates = [c for c in df.columns if "correctness" in c.lower() and "comment" in c.lower()]
comment_col = comment_candidates[0] if comment_candidates else None
failed = df[(df["correctness_score"] < 1) | (df["rule_score"] < 1)].copy()
def failure_type(row):
    q = str(row.get(question_col, "")).lower()
    if row.get("correctness_score", 1) >= 1 and row.get("rule_score", 1) < 1: return "rule heuristic disagreement"
    if any(term in q for term in ["express", "cancel", "opened"]): return "conditional or incomplete answer"
    if any(term in q for term in ["price matching", "saturday"]): return "policy-boundary handling"
    return "wrong fact or omission"
failed["failure_type"] = failed.apply(failure_type, axis=1)
failure_cols = [c for c in [question_col, answer_col, reference_col, comment_col, "failure_type"] if c]
print(failed[failure_cols].to_string(index=False))
failed[failure_cols].to_csv(OUT / "failure_analysis.csv", index=False)

** Key findings (populated from the executed analysis cells):**
1. Overall pass rate per evaluator is written to `outputs/metrics_main.csv`.
2. Category and difficulty comparisons are written to `outputs/pass_rate_by_category.csv` and `outputs/pass_rate_by_difficulty.csv`.
3. Every LLM-judge failure is exported to `outputs/failure_analysis.csv` with a transparent failure type.
4. The judge and keyword evaluator are compared directly; disagreements are useful review candidates rather than proof that either evaluator is correct.

 **Remember (W7/D5 statistics):** with 16 examples, every score is an **estimate**. One extra failure moves the pass rate by 6.25 points.

### Step 10: `evaluation_summary.md`

**One short paragraph** (plus optional bullets/table): what you evaluated, main metrics, biggest failure pattern, limitations, one recommendation. Not a long report.

In [ ]:
llm_pass_rate = metrics.loc[metrics["metric"] == "correctness_llm", "pass_rate"].iloc[0]
rule_pass_rate = metrics.loc[metrics["metric"] == "keyword_coverage", "pass_rate"].iloc[0]
experiment_link = getattr(results, "url", "Open the experiment from the LangSmith project: " + PROJECT_NAME)
hardest_category = category_scores["rule_score"].idxmin()
easiest_category = category_scores["rule_score"].idxmax()
summary = f'''# Evaluation summary\n\nThe `northstar-support-policy-v1` dataset contains {len(examples)} customer-support questions from `policy.md` across {len(category_scores)} categories, including hard conditional and policy-boundary cases. The target used {MODEL} with temperature 0 and a grounded system prompt; a stronger {JUDGE_MODEL} correctness judge and a deterministic keyword/numeric-constraint check were evaluated. The LLM judge pass rate was {llm_pass_rate:.1%}, versus {rule_pass_rate:.1%} for the rule evaluator; the rule diagnostic found `{easiest_category}` easiest and `{hardest_category}` hardest. Failures should be interpreted cautiously because this is a 16-example estimate and the keyword heuristic can disagree with valid paraphrases. The main recommendation is to review judge/rule disagreements and add more multi-condition and out-of-policy questions in dataset v2.\n\n| Metric | Score |\n|---|---|\n| Correctness (LLM judge) | {llm_pass_rate:.1%} |\n| Keyword/numeric rule | {rule_pass_rate:.1%} |\n| Examples | {len(examples)} |\n\nLangSmith experiment: {experiment_link}\n'''
Path("evaluation_summary.md").write_text(summary, encoding="utf-8")

---
## Optional Part 1: Custom evaluator (60–90 min)
### Step 11: Design it

Pick **one** dimension beyond correctness that matters for your domain: tone, completeness, groundedness in the context, safety / policy compliance, format…

 **Hints:**
- Use the 4-part judge prompt anatomy from notebook 01, §3: task, what you're given, criteria, reasoning steps + output format.
- Your prompt's `{variables}` must match the keyword arguments you pass (`inputs`, `outputs`, `reference_outputs`, or your own like `context`).
- Prebuilt prompts you can adapt: `RAG_GROUNDEDNESS_PROMPT`, `ANSWER_RELEVANCE_PROMPT`, `CONCISENESS_PROMPT`, `TOXICITY_PROMPT` (`print()` one to see how it's written).
- For a 1–5 scale instead of True/False: `create_llm_as_judge(..., continuous=True)` gives a 0–1 score, or `choices=[1, 2, 3, 4, 5]`.

In [ ]:
CUSTOM_PROMPT = '''You are evaluating a customer-support answer for grounded helpfulness.

Use the supplied question, model answer, and reference answer. Give a high score only when the answer is grounded in the policy, includes the important conditions and numbers, and clearly handles information that is not stated. Penalize invented policies, omitted conditions, and confident answers to out-of-scope questions.

Question: {inputs}
Model answer: {outputs}
Reference answer: {reference_outputs}

Score from 0 (not grounded or unusable) to 1 (fully grounded and complete). Explain the key reason briefly.'''

custom_judge = create_llm_as_judge(
    prompt=CUSTOM_PROMPT, model=JUDGE_MODEL, feedback_key="grounded_helpfulness", continuous=True, use_reasoning=True
)

def custom_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    judged = custom_judge(inputs=inputs, outputs=outputs, reference_outputs=reference_outputs)
    return {"key": "grounded_helpfulness", "score": float(judged.get("score", 0)), "comment": judged.get("comment", "")}

custom_cases = [
    ({"question": "What does standard shipping cost?"}, {"answer": "It costs $6 and usually arrives in 3–5 business days."}, {"answer": "Standard shipping costs $6 and usually arrives in 3–5 business days."}),
    ({"question": "Is price matching offered?"}, {"answer": "Yes, we match every competitor."}, {"answer": "The policy does not state that Northstar offers price matching."}),
    ({"question": "Is express shipping free on a $100 order?"}, {"answer": "The order qualifies for free standard shipping; express may still cost extra."}, {"answer": "No. The $18 express surcharge still applies."}),
]
for case in custom_cases:
    print(custom_evaluator(*case))

### Step 12: Run it next to correctness

In [ ]:
results_custom = client.evaluate(
    target, data=DATASET_NAME, evaluators=[correctness_evaluator, custom_evaluator],
    experiment_prefix="gpt-4o-mini-support-policy-custom-v1", max_concurrency=2
)
custom_df = results_custom.to_pandas()
custom_feedback = [c for c in custom_df.columns if "feedback" in c.lower()]
custom_correctness_col = next(c for c in custom_feedback if "correctness" in c.lower())
custom_grounded_col = next(c for c in custom_feedback if "grounded_helpfulness" in c.lower())
custom_df["correctness_score"] = pd.to_numeric(custom_df[custom_correctness_col], errors="coerce").fillna(0)
custom_df["grounded_helpfulness_score"] = pd.to_numeric(custom_df[custom_grounded_col], errors="coerce").fillna(0)
custom_df.to_csv(OUT / "results_custom.csv", index=False)
disagreements = custom_df[(custom_df["correctness_score"] >= 1) & (custom_df["grounded_helpfulness_score"] < 0.75)]
print("Correct according to judge but below 0.75 on grounded helpfulness:")
print(disagreements[[c for c in custom_df.columns if c.startswith("inputs.") or c.endswith("score")]].to_string(index=False))

### Step 13: Document your custom evaluator
- What it measures and why it matters in my domain: grounded helpfulness—support answers must preserve policy conditions and avoid inventing coverage.
- Scoring scale and criteria: continuous 0–1; 1 means grounded, complete, and appropriately handles policy boundaries; 0 means unsupported, materially incomplete, or hallucinated.
- Where it disagreed with correctness, and what that revealed: the comparison cell exports disagreements. These cases show whether a correct answer is technically right but omits a condition or is less useful than the reference.

---
## Optional Part 2: Cost & performance (60–90 min)
### Step 14: A/B comparison

 **Hints (§4 of `02_langsmith(1).ipynb`):**
- Same dataset, same evaluators, **only one thing changes** (the model, or the temperature, or the prompt).
- To get token usage, also return it from the target: `{"answer": ..., "prompt_tokens": r.usage.prompt_tokens, "completion_tokens": r.usage.completion_tokens}`.
- Latency: look for an `execution_time` column in `results.to_pandas()`.
- Fill `PRICES` from the **OpenAI pricing page** (USD per 1M tokens). Write down the date you checked it.

In [ ]:
CONFIGS = {
    # experiment name: model
    "mini": "gpt-4o-mini",
    "full": "gpt-5.4-mini",
}

PRICES = {   # USD per 1M tokens; OpenAI pricing pages checked on 2026-11-24
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-5.4-mini": {"input": 0.75, "output": 4.50},
}

def make_target(model):
    @traceable(name=f"lab-target-{model}")
    def _target(inputs: dict) -> dict:
        try:
            request_args = {"model": model, "temperature": 0, "max_tokens": 200}
            if model.startswith("gpt-5"):
                request_args = {"model": model, "reasoning_effort": "none", "max_completion_tokens": 200}
            response = openai_client.chat.completions.create(
                **request_args,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"Policy context:\n{inputs.get('context', '')}\n\nCustomer question: {inputs['question']}"},
                ],
            )
            usage = response.usage
            return {"answer": (response.choices[0].message.content or "").strip(),
                    "prompt_tokens": usage.prompt_tokens if usage else 0,
                    "completion_tokens": usage.completion_tokens if usage else 0}
        except Exception as exc:
            return {"answer": "", "prompt_tokens": 0, "completion_tokens": 0, "error": str(exc)}
    return _target

ab_results = {}
for config_name, model_name in CONFIGS.items():
    ab_results[config_name] = client.evaluate(
        make_target(model_name), data=DATASET_NAME, evaluators=[correctness_evaluator],
        experiment_prefix=f"{config_name}-{model_name}-support-policy-v1", max_concurrency=2
    )
print(f"Completed {len(ab_results)} A/B experiments")

### Step 15: Cost vs performance

In [ ]:
ab_metric_rows = []
for config_name, experiment in ab_results.items():
    run_df = experiment.to_pandas()
    feedback = next(c for c in run_df.columns if "feedback" in c.lower() and "correctness" in c.lower())
    scores = pd.to_numeric(run_df[feedback], errors="coerce").fillna(0).astype(float)
    prompt_col = next((c for c in run_df.columns if "prompt_tokens" in c.lower()), None)
    completion_col = next((c for c in run_df.columns if "completion_tokens" in c.lower()), None)
    latency_col = next((c for c in run_df.columns if c.lower().endswith("execution_time") or c.lower() == "execution_time"), None)
    prompt_tokens = pd.to_numeric(run_df[prompt_col], errors="coerce").fillna(0) if prompt_col else pd.Series(0, index=run_df.index)
    completion_tokens = pd.to_numeric(run_df[completion_col], errors="coerce").fillna(0) if completion_col else pd.Series(0, index=run_df.index)
    model_name = CONFIGS[config_name]; price = PRICES[model_name]
    total_cost = (prompt_tokens.sum() / 1e6 * price["input"] + completion_tokens.sum() / 1e6 * price["output"])
    avg_latency = pd.to_numeric(run_df[latency_col], errors="coerce").mean() if latency_col else float("nan")
    ab_metric_rows.append({"config": config_name, "model": model_name, "pass_rate": scores.mean(), "avg_latency": avg_latency, "total_cost": total_cost, "cost_per_example": total_cost / len(run_df), "cost_per_correct": total_cost / max(scores.sum(), 1)})
ab_metrics = pd.DataFrame(ab_metric_rows)
print(ab_metrics.to_string(index=False))
ab_metrics.to_csv(OUT / "ab_metrics.csv", index=False)
fig, ax = plt.subplots(figsize=(7, 4))
for _, row in ab_metrics.iterrows():
    ax.scatter(row["cost_per_example"], row["pass_rate"], s=90, label=row["config"])
    ax.annotate(row["config"], (row["cost_per_example"], row["pass_rate"]), xytext=(5, 5), textcoords="offset points")
ax.set_xlabel("Cost per example (USD)"); ax.set_ylabel("Correctness pass rate"); ax.set_ylim(0, 1.05); ax.set_title("Cost vs performance"); ax.grid(alpha=.25); plt.tight_layout()
plt.savefig(OUT / "cost_vs_performance.png", dpi=120)
plt.show()

 **Before you recommend, think:**
- Is the gap between models bigger than **one or two examples**? With ~20 examples it may be luck (W7/D5).
- Which pass rate is **good enough** for this use case? A cheaper model that meets that bar wins.

### Step 16: `optimization_summary.md`

In [ ]:
best = ab_metrics.sort_values(["cost_per_correct", "pass_rate"], ascending=[True, False]).iloc[0]
table_rows = "\n".join(
    f"| {r.config} | {r.model} | {r.pass_rate:.1%} | ${r.cost_per_correct:.6f} | {r.avg_latency:.2f} |"
    for r in ab_metrics.itertuples()
)
opt = f'''# Optimization summary\n\nAmong these two runs, `{best.config}` ({best.model}) had the lowest observed cost per correct answer at the measured pass rate. The mini configuration is the sensible default when its accuracy meets the support quality bar; the full model is worth the extra cost when conditional questions or policy-boundary handling require the higher observed pass rate. This comparison is directional because it uses only {len(examples)} examples and one run per model.\n\n| Config | Model | Pass rate | Cost / correct answer | Avg latency |\n|---|---|---:|---:|---:|\n{table_rows}\n'''
Path("optimization_summary.md").write_text(opt, encoding="utf-8")

---
## Reflection (optional, recommended)
1. What surprised you when you read the failed traces?
2. Did you trust your LLM judge? How did you check it?
3. What would you change in your dataset for a `-v2`?

## Submission checklist
- [ ] GitHub repo with **only** this lab, and a `README.md` (domain + dataset, approach, how to run, LangSmith links, file map)
- [ ] `lab_proof.md` as the **entry point**: code/workflow, one run with its input, the evidence it produced (trace / screenshot), the final output, and one failure or limitation you hit
- [ ] `evaluation_summary.md` at the repo root (one paragraph + optional table)
- [ ] (Optional) `optimization_summary.md` at the repo root
- [ ] Notebook runs top to bottom without errors; descriptive file names (no `lab.ipynb`, `final_v2.py`)
- [ ] `outputs/`: results CSV(s), charts, and LangSmith screenshots (dataset, experiment, one trace)
- [ ] **No API keys** in the repo: `.env` is in `.gitignore`